In [1]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_2'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2101, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 2101 (delta 11), reused 15 (delta 3), pack-reused 2066 (from 1)
Receiving objects: 100% (2101/2101), 263.06 MiB | 20.03 MiB/s, done.
Resolving deltas: 100% (400/400), done.
Updating files: 100% (1348/1348), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_2


In [2]:
!pip install -q awscli
import os
import json

# ==========================================
# 1. LOAD AWS CREDENTIALS FROM COLAB SECRETS
# ==========================================
def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "us-east-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region
os.environ["AWS_DEFAULT_OUTPUT"] = "json"

print(f"AWS credentials loaded. Region: {_aws_region}")

# ==========================================
# 2. RUN NON-INTERACTIVE IAM ROLE AUTOMATION
# ==========================================
!echo "--- 1. Testing AWS Credentials ---"
!aws sts get-caller-identity

!echo "--- 2. Generating Temporary SageMaker Trust Policy ---"
trust_policy = {
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": { "Service": "sagemaker.amazonaws.com" },
      "Action": "sts:AssumeRole"
    }
  ]
}

with open("sagemaker-trust-policy.json", "w") as f:
    json.dump(trust_policy, f)

!echo "--- 3. Creating Custom IAM Roles ---"
!aws iam create-role --role-name Lesson6CleanExecRole --assume-role-policy-document file://sagemaker-trust-policy.json || true
!aws iam create-role --role-name model-engineering-lab-sagemaker-execution --assume-role-policy-document file://sagemaker-trust-policy.json || true
!aws iam create-role --role-name SageMakerNeoLabRole --assume-role-policy-document file://sagemaker-trust-policy.json || true
!aws iam create-role --role-name unified-mlops-mlflow-dev-execution-role --assume-role-policy-document file://sagemaker-trust-policy.json || true

!echo "--- 4. Attaching AWS Managed Policies ---"
!aws iam attach-role-policy --role-name Lesson6CleanExecRole --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
!aws iam attach-role-policy --role-name model-engineering-lab-sagemaker-execution --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
!aws iam attach-role-policy --role-name SageMakerNeoLabRole --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess

# Attach both SageMaker and S3 permissions for the MLOps dev role
!aws iam attach-role-policy --role-name unified-mlops-mlflow-dev-execution-role --policy-arn arn:aws:iam::aws:policy/AmazonSageMakerFullAccess
!aws iam attach-role-policy --role-name unified-mlops-mlflow-dev-execution-role --policy-arn arn:aws:iam::aws:policy/AmazonS3FullAccess

!echo "--- 5. Cleanup Temporary Files ---"
!rm -f sagemaker-trust-policy.json

!echo "--- 6. Verification ---"
!aws iam get-role --role-name unified-mlops-mlflow-dev-execution-role --query "Role.[RoleName, Arn]" --output text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 36.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
AWS credentials loaded. Region: us-east-1
--- 1. Testing AWS Credentials ---
{
    "UserId": "455865672536",
    "Account": "455865672536",
    "Arn": "arn:aws:iam::455865672536:root"
}
--- 2. Generating Temporary SageMaker Trust Policy ---
--- 3. Creating Custom IAM Roles ---

An error occurred (EntityAlreadyExists) when calling the CreateRole operation: Role with name Lesson6CleanExecRole already exists.

An error occurred (EntityAlreadyExists) when calling the CreateRole operation: Role with name model-engineering-lab-sagemaker-execution already exists.

An erro

In [3]:
# Create the S3 buckets this notebook uses (idempotent - only creates missing ones).
!pip install -q boto3
import boto3
from botocore.exceptions import ClientError
import os

# 1. Initialize S3 & STS clients using your loaded region
_region = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-east-1"
_s3 = boto3.client("s3", region_name=_region)
_sts = boto3.client("sts", region_name=_region)

# 2. Get your AWS Account ID to ensure global bucket uniqueness
try:
    account_id = _sts.get_caller_identity()["Account"]
except Exception as e:
    raise RuntimeError(f"Failed to authenticate with AWS credentials: {e}")

# 3. Define globally unique bucket names (Appends Account ID)
BASE_BUCKETS = ['usecase-etl-1']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKETS]

CREATED_BUCKETS = []

for _b in REQUIRED_BUCKETS:
    bucket_successfully_created_or_verified = False
    try:
        print(f"Attempting to create globally unique bucket: {_b} in region: {_region}")
        if _region != "us-east-1":
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={'LocationConstraint': _region}
            )
        else:
            _s3.create_bucket(Bucket=_b)

        CREATED_BUCKETS.append(_b)
        print(f"✅ Successfully created bucket: {_b}")
        bucket_successfully_created_or_verified = True

    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyOwnedByYou", "Conflict"):
            print(f"ℹ️ Bucket {_b} already owned by your account. Verifying accessibility...")
            try:
                _s3.head_bucket(Bucket=_b)
                print(f"✅ Bucket {_b} verified as accessible.")
                bucket_successfully_created_or_verified = True
            except ClientError as head_e:
                print(f"❌ ERROR accessing existing bucket {_b}: {head_e}")
        elif _code == "BucketAlreadyExists":
            print(f"❌ FATAL: Name collision! Bucket {_b} is globally owned by another AWS user.")
        else:
            print(f"❌ Could not create bucket {_b} ({_code}): {_e}")

    if not bucket_successfully_created_or_verified:
        raise RuntimeError(f"FATAL: S3 bucket {_b} is not ready. Please check AWS credentials, region, and bucket permissions.")

print("Buckets ready:", REQUIRED_BUCKETS)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 3.4 MB/s eta 0:00:00
Attempting to create globally unique bucket: usecase-etl-1-455865672536 in region: us-east-1
✅ Successfully created bucket: usecase-etl-1-455865672536
Buckets ready: ['usecase-etl-1-455865672536']


# Notebook 2 — Data Cleaning & Transformation

**Use Case 2 goal:** take the exploration-ready retail file and turn it into a cleaner, reporting-friendly dataset.

## What learners should understand
- Why data cleaning is separated from raw ingestion
- Why each transformation is performed
- How the cleaned file becomes the input for the ETL aggregation step


## Dependencies, AWS setup, and files used

### Python packages
- **boto3** — used to read the Use Case 1 handoff file from Amazon S3 and write the cleaned dataset back to S3
- **pandas** — used for learner-friendly transformation logic before we translate the ETL idea into Glue PySpark
- **io** — used to move CSV content between S3 and pandas without extra manual downloads
- **pathlib** — kept as a local rehearsal fallback


In [4]:
# AWS-friendly setup cell for SageMaker Studio or any Jupyter environment with AWS credentials.
from pathlib import Path
from io import BytesIO, StringIO
import os
import boto3
import pandas as pd

AWS_REGION = os.getenv('AWS_REGION', boto3.session.Session().region_name or 'us-east-1')

# Dynamically set S3_BUCKET from the REQUIRED_BUCKETS variable set in the bucket-create cell
# This ensures we use the globally unique bucket name with the account ID suffix.
S3_BUCKET = REQUIRED_BUCKETS[0] if 'REQUIRED_BUCKETS' in globals() and REQUIRED_BUCKETS else "usecase-etl-1"

# ✅ FIX: Disable fallback since you're using real S3
USE_LOCAL_FALLBACK = False

s3_client = boto3.client('s3', region_name=AWS_REGION)


def parse_s3_uri(uri: str):
    bucket, key = uri.replace('s3://', '', 1).split('/', 1)
    return bucket, key


def read_csv_aws_first(s3_uri: str, local_path: Path) -> pd.DataFrame:
    """Try S3 first, fallback to local if it fails."""
    try:
        bucket, key = parse_s3_uri(s3_uri)
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        print("✅ Reading from S3:", s3_uri)
        return pd.read_csv(BytesIO(obj['Body'].read()))
    except Exception as e:
        print(f"⚠️ S3 read failed: {e}")
        print("📂 Falling back to local file:", local_path)
        return pd.read_csv(local_path)


def write_csv_aws_first(df: pd.DataFrame, s3_uri: str, local_path: Path) -> None:
    """Write to S3, fallback to local if needed."""
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    try:
        bucket, key = parse_s3_uri(s3_uri)
        s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue().encode('utf-8'))
        print("✅ Written to S3:", s3_uri)
    except Exception as e:
        print(f"⚠️ S3 write failed: {e}")
        print("📂 Writing locally instead:", local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_text(csv_buffer.getvalue(), encoding='utf-8')


LOCAL_INPUT_PATH = Path('./retail_exploration_ready.csv')
LOCAL_OUTPUT_PATH = Path('./retail_cleaned.csv')

# Update S3 URIs to use the dynamically set S3_BUCKET
INPUT_S3_URI = f"s3://{S3_BUCKET}/processed/retail_exploration_ready.csv"
OUTPUT_S3_URI = f"s3://{S3_BUCKET}/processed/retail_cleaned.csv"

print('Input source:', INPUT_S3_URI)
print('Output target:', OUTPUT_S3_URI)

Input source: s3://usecase-etl-1-455865672536/processed/retail_exploration_ready.csv
Output target: s3://usecase-etl-1-455865672536/processed/retail_cleaned.csv


## Step 1 — Read the exploration-ready file

### Why this step is performed
This notebook continues the same story from Use Case 1.

### Expected result
A DataFrame that is ready for formal cleaning rules.


In [5]:
df = read_csv_aws_first(INPUT_S3_URI, LOCAL_INPUT_PATH)
print('Shape before cleaning:', df.shape)
display(df.head())


✅ Reading from S3: s3://usecase-etl-1-455865672536/processed/retail_exploration_ready.csv
Shape before cleaning: (500, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateParsed
0,536365,71053,WHITE METAL LANTERN,6,02/01/2011 11:08,5.49,17889.0,Belgium,2011-02-01 11:08:00
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,01/28/2011 11:32,4.22,16943.0,Germany,2011-01-28 11:32:00
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,02/05/2011 08:48,5.80,18065.0,Netherlands,2011-02-05 08:48:00
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,01/13/2011 13:54,7.55,14512.0,United Kingdom,2011-01-13 13:54:00
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,03/22/2011 17:56,4.03,17075.0,Germany,2011-03-22 17:56:00


## Step 2 — Convert dates and derive reporting fields

### Why this step is performed
Most analytics and ETL outputs require usable time fields such as transaction date, month, and year.


In [6]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['TransactionDate'] = df['InvoiceDate'].dt.date.astype(str)
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)

display(df[['InvoiceDate', 'TransactionDate', 'Year', 'Month']].head())


,InvoiceDate,TransactionDate,Year,Month
0,2011-02-01 11:08:00,2011-02-01,2011,2011-02
1,2011-01-28 11:32:00,2011-01-28,2011,2011-01
2,2011-02-05 08:48:00,2011-02-05,2011,2011-02
3,2011-01-13 13:54:00,2011-01-13,2011,2011-01
4,2011-03-22 17:56:00,2011-03-22,2011,2011-03


In [7]:
(df.columns)


Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'InvoiceDateParsed',
       'TransactionDate', 'Year', 'Month'],
      dtype='object')

## Step 3 — Handle missing values and remove invalid records

### Why this step is performed
Not every row should move forward into reporting. Here we apply simple, teachable business rules.


In [8]:
df['Description'] = df['Description'].fillna('UNKNOWN_ITEM')
df['CustomerID'] = df['CustomerID'].fillna('UNKNOWN_CUSTOMER')

df = df[df['InvoiceDate'].notna()]
df = df[df['UnitPrice'] > 0]

print('Shape after cleaning:', df.shape)


Shape after cleaning: (489, 12)


## Step 4 — Engineer business metrics

### Why this step is performed
Learners should see exactly where business fields such as revenue come from.


In [9]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['IsReturn'] = df['Quantity'] < 0

display(df[['Quantity', 'UnitPrice', 'Revenue', 'IsReturn']].head())


,Quantity,UnitPrice,Revenue,IsReturn
0,6,5.49,32.94,False
1,2,4.22,8.44,False
2,6,5.80,34.80,False
3,4,7.55,30.20,False
4,6,4.03,24.18,False


## Step 5 — Save the cleaned dataset

### Why this step is performed
This cleaned file becomes the ETL input for Notebook 3.


In [10]:
write_csv_aws_first(df, OUTPUT_S3_URI, LOCAL_OUTPUT_PATH)
print('Cleaned dataset saved to:')
print(OUTPUT_S3_URI if not USE_LOCAL_FALLBACK else LOCAL_OUTPUT_PATH.resolve())


✅ Written to S3: s3://usecase-etl-1-455865672536/processed/retail_cleaned.csv
Cleaned dataset saved to:
s3://usecase-etl-1-455865672536/processed/retail_cleaned.csv


In [11]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))


Current Time in IST: 2026-09-14 09:00:56
